# Task 2.1: Find Common Edges (pySCENIC ↔ scPRINT2)

**Goal:** Compare edges across pySCENIC-Net and scPRINT2-Net to assess method agreement.

**Approach:**
1. Load pySCENIC edge list (motif-pruned)
2. Load scPRINT2 GRN h5ad files (4 cell states)
3. Extract edges from scPRINT2 attention matrices
4. Compute pairwise edge overlap per state
5. Identify top shared edges and TF agreement

**Outputs:**
- `edge_overlap_scenic_vs_scprint2.csv` — overlap statistics per state
- Per-state shared edge lists

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pandas as pd
import numpy as np
import scanpy as sc
import gc

DRIVE_BASE = '/content/drive/MyDrive/benchmarking_paper/datasets'
DRIVE_PYSCENIC = os.path.join(DRIVE_BASE, 'pyscenic')
DRIVE_SCPRINT = os.path.join(DRIVE_BASE, 'scPRINT_grn_outputs_final')
DRIVE_ANALYSIS = os.path.join(DRIVE_BASE, 'analysis_outputs/may20')

os.makedirs(DRIVE_ANALYSIS, exist_ok=True)

STATES = ['CD4_rest', 'CD4_act', 'CD8_rest', 'CD8_act']

print(f'pySCENIC dir: {DRIVE_PYSCENIC}')
print(f'scPRINT2 dir: {DRIVE_SCPRINT}')
print(f'Analysis output dir: {DRIVE_ANALYSIS}')
print(f'Cell states to compare: {STATES}')

pySCENIC dir: /content/drive/MyDrive/benchmarking_paper/datasets/pyscenic
scPRINT2 dir: /content/drive/MyDrive/benchmarking_paper/datasets/scPRINT_grn_outputs_final
Analysis output dir: /content/drive/MyDrive/benchmarking_paper/datasets/analysis_outputs/may20
Cell states to compare: ['CD4_rest', 'CD4_act', 'CD8_rest', 'CD8_act']


## Step 1: Load pySCENIC Edge List

In [ ]:
# Load pySCENIC edge list
pyscenic_path = os.path.join(DRIVE_PYSCENIC, 'pyscenic_net_edges.csv')
print(f'Loading pySCENIC edges from: {pyscenic_path}')
pyscenic_edges = pd.read_csv(pyscenic_path)

print(f'pySCENIC edge list shape: {pyscenic_edges.shape}')
print(f'Columns: {list(pyscenic_edges.columns)}')
print(f'\nFirst few rows:')
print(pyscenic_edges.head())

# Create (TF, target) tuple set for fast lookup
pyscenic_edge_set = set(zip(pyscenic_edges['tf'], pyscenic_edges['target']))
print(f'\nTotal pySCENIC edges: {len(pyscenic_edge_set)}')
print(f'Unique TFs: {pyscenic_edges["tf"].nunique()}')
print(f'Unique targets: {pyscenic_edges["target"].nunique()}')

Loading pySCENIC edges from: /content/drive/MyDrive/benchmarking_paper/datasets/pyscenic/pyscenic_net_edges.csv
pySCENIC edge list shape: (27019, 3)
Columns: ['tf', 'target', 'importance']

First few rows:
       tf      target  importance
0  ARID3A       APPL2    0.356992
1  ARID3A     ARHGAP1    0.287484
2  ARID3A     ARHGEF7    1.273510
3  ARID3A  ENTPD1-AS1    0.528954
4  ARID3A       GPAA1    0.039146

Total pySCENIC edges: 27019
Unique TFs: 235
Unique targets: 9518


## Step 2: Load scPRINT2 GRN Files and Extract Edges

Extract edges from attention matrices using a threshold. scPRINT2 outputs attention weights; we keep TF→target pairs with high attention.

In [ ]:
# Per-TF target count distribution
row_idxs, col_idxs = np.where(mat > 0.001)
tf_filter = is_tf[row_idxs]
tfs_sym = symbols[row_idxs[tf_filter]]
targets_sym = symbols[col_idxs[tf_filter]]

from collections import Counter
targets_per_tf = Counter(tfs_sym)
print("Targets per TF distribution:")
counts = sorted(targets_per_tf.values())
print(f"  min: {min(counts)}, median: {np.median(counts):.0f}, max: {max(counts)}, mean: {np.mean(counts):.1f}")
print(f"  TFs with >10 targets: {sum(1 for c in counts if c > 10)}")
print(f"  TFs with >50 targets: {sum(1 for c in counts if c > 50)}")
print(f"\nTop 10 TFs by target count:")
for tf, n in Counter(tfs_sym).most_common(10):
    print(f"  {tf}: {n}")
print(f"\nCanonical TFs:")
for tf in ['KLF2','TCF7','BATF','IRF4','FOXO1','JUN']:
    n = targets_per_tf.get(tf, 0)
    print(f"  {tf}: {n} targets")

Targets per TF distribution:
  min: 26, median: 40, max: 63, mean: 40.7
  TFs with >10 targets: 982
  TFs with >50 targets: 51

Top 10 TFs by target count:
  HMG20A: 63
  FOSB: 62
  POU2F1: 62
  ARID5A: 62
  ZNF490: 61
  CEBPD: 59
  EGR1: 57
  NR1D1: 57
  GTF2IRD2B: 57
  ZNF530: 57

Canonical TFs:
  KLF2: 36 targets
  TCF7: 37 targets
  BATF: 35 targets
  IRF4: 50 targets
  FOXO1: 33 targets
  JUN: 39 targets


In [ ]:
# DIRECTIONALITY CONFIRMATION — save result
grn = sc.read_h5ad(os.path.join(DRIVE_SCPRINT, 'CD4_rest_grn_direct.h5ad'))
mat = grn.varp['GRN']
if hasattr(mat, 'toarray'):
    mat = mat.toarray()

gene_names = np.array(grn.var_names)
symbols    = grn.var['symbol'].values
is_tf      = grn.var['isTF'].values
ens2sym    = dict(zip(grn.var_names, grn.var['symbol'])) if 'symbol' in grn.var.columns else {}

results = {}
for orientation, (tf_axis, target_axis) in {
    'row=TF, col=target': ('row', 'col'),
    'col=TF, row=target': ('col', 'row'),
}.items():
    row_idxs, col_idxs = np.where(mat > 0.001)  # lower threshold for directionality test
    tf_idxs     = row_idxs if tf_axis == 'row' else col_idxs
    target_idxs = col_idxs if tf_axis == 'row' else row_idxs

    tf_filter   = is_tf[tf_idxs]
    tf_idxs     = tf_idxs[tf_filter]
    target_idxs = target_idxs[tf_filter]

    tfs_sym     = symbols[tf_idxs]
    targets_sym = symbols[target_idxs]
    not_self    = tfs_sym != targets_sym
    tfs_sym     = tfs_sym[not_self]
    targets_sym = targets_sym[not_self]

    edge_set    = set(zip(tfs_sym, targets_sym))
    overlap     = len(edge_set & pyscenic_edge_set)
    n_tfs       = len(set(tfs_sym))
    n_targets   = len(set(targets_sym))

    results[orientation] = overlap
    print(f"{orientation}: {len(edge_set):,} edges | {n_tfs} TFs | {n_targets} targets | pySCENIC overlap: {overlap}")

winner = max(results, key=results.get)
print(f"\n✓ Confirmed orientation: {winner}")

# Save decision
with open(os.path.join(DRIVE_ANALYSIS, 'directionality_decision.txt'), 'w') as f:
    f.write(f"Confirmed orientation: {winner}\n")
    f.write(f"Basis: higher pySCENIC edge overlap on CD4_rest\n")
    for k, v in results.items():
        f.write(f"  {k}: {v} overlapping edges\n")
print("Saved to Drive ✓")

row=TF, col=target: 39,906 edges | 982 TFs | 289 targets | pySCENIC overlap: 96
col=TF, row=target: 28,522 edges | 202 TFs | 1404 targets | pySCENIC overlap: 11

✓ Confirmed orientation: row=TF, col=target
Saved to Drive ✓


In [ ]:
scprint_edges_by_state = {}

for state in STATES:
    csv_path = os.path.join(DRIVE_SCPRINT, f'{state}_edges_direct.csv')
    if not os.path.exists(csv_path):
        print(f'WARNING: {csv_path} not found')
        continue

    df = pd.read_csv(csv_path)
    print(f'{state}: {df.shape[0]} edges | columns: {df.columns.tolist()}')
    print(df.head(3))
    print()

CD4_rest: 9470 edges | columns: ['TF', 'target', 'weight', 'state']
     TF   target    weight     state
0  NFYA    ASH1L  0.019261  CD4_rest
1  NFYA  ZSCAN29  0.020794  CD4_rest
2  NFYA    CEBPG  0.011975  CD4_rest

CD4_act: 6904 edges | columns: ['TF', 'target', 'weight', 'state']
     TF   target    weight    state
0  NFYA    ASH1L  0.028228  CD4_act
1  NFYA   ZNF549  0.011812  CD4_act
2  NFYA  ZSCAN29  0.020491  CD4_act

CD8_rest: 7873 edges | columns: ['TF', 'target', 'weight', 'state']
     TF   target    weight     state
0  NFYA    ASH1L  0.026771  CD8_rest
1  NFYA  ZSCAN29  0.020347  CD8_rest
2  NFYA    RPL10  0.011475  CD8_rest

CD8_act: 8446 edges | columns: ['TF', 'target', 'weight', 'state']
     TF   target    weight    state
0  NFYA    ASH1L  0.030389  CD8_act
1  NFYA  ZSCAN29  0.021027  CD8_act
2  NFYA      B2M  0.031734  CD8_act



In [ ]:
scprint_edges_by_state = {}

for state in STATES:
    csv_path = os.path.join(DRIVE_SCPRINT, f'{state}_edges_direct.csv')
    df = pd.read_csv(csv_path)
    edge_set = set(zip(df['TF'], df['target']))
    scprint_edges_by_state[state] = edge_set
    print(f'{state}: {len(edge_set):,} edges | {df["TF"].nunique()} unique TFs | {df["target"].nunique()} unique targets')

CD4_rest: 9,470 edges | 982 unique TFs | 51 unique targets
CD4_act: 6,904 edges | 973 unique TFs | 45 unique targets
CD8_rest: 7,873 edges | 999 unique TFs | 43 unique targets
CD8_act: 8,446 edges | 1017 unique TFs | 46 unique targets


## Step 3: Compute Edge Overlap per State

In [ ]:
df = pd.read_csv(os.path.join(DRIVE_SCPRINT, 'CD4_rest_edges_direct.csv'))
scprint_tfs = set(df['TF'].unique())
scprint_targets = set(df['target'].unique())

print("=== FORMAT CHECK ===")
print(f"pySCENIC TF examples: {sorted(scenic_tfs)[:10]}")
print(f"scPRINT TF examples: {sorted(scprint_tfs)[:10]}")
print(f"\npySCENIC target examples: {sorted(scenic_targets)[:10]}")
print(f"scPRINT target examples: {sorted(scprint_targets)[:10]}")

print("\n=== OVERLAP POTENTIAL ===")
print(f"Shared TF names: {len(scenic_tfs & scprint_tfs)}")
print(f"Shared target names: {len(scenic_targets & scprint_targets)}")

=== FORMAT CHECK ===
pySCENIC TF examples: ['ARID3A', 'ARNTL', 'ATF1', 'ATF2', 'ATF3', 'ATF4', 'ATF5', 'ATF6', 'ATF6B', 'BACH1']
scPRINT TF examples: ['ADNP', 'ADNP2', 'AEBP1', 'AEBP2', 'AHCTF1', 'AHDC1', 'AHR', 'AKAP8', 'AKAP8L', 'AKNA']

pySCENIC target examples: ['A2MP1', 'AAAS', 'AAGAB', 'AAK1', 'AAMDC', 'AAMP', 'AAR2', 'AARS', 'AARS2', 'AASDH']
scPRINT target examples: ['AHNAK', 'AMICA1', 'ASH1L', 'B2M', 'CCL3', 'CCL4', 'CEBPG', 'EEF1A1', 'EEF1B2', 'EVL']

=== OVERLAP POTENTIAL ===
Shared TF names: 180
Shared target names: 37


In [ ]:
# Check: how many targets does scPRINT have at different thresholds?
grn = sc.read_h5ad(os.path.join(DRIVE_SCPRINT, 'CD4_rest_grn_direct.h5ad'))
mat = grn.varp['GRN']
if hasattr(mat, 'toarray'):
    mat = mat.toarray()

symbols = grn.var['symbol'].values
is_tf   = grn.var['isTF'].values

for thresh in [0.001, 0.0005, 0.0001, 0.00005]:
    row_idxs, col_idxs = np.where(mat > thresh)
    tf_filter   = is_tf[row_idxs]
    tfs_sym     = symbols[row_idxs[tf_filter]]
    targets_sym = symbols[col_idxs[tf_filter]]
    not_self    = tfs_sym != targets_sym
    targets_sym = targets_sym[not_self]
    tfs_sym     = tfs_sym[not_self]

    shared_targets = len(set(targets_sym) & scenic_targets)
    print(f"thresh={thresh}: {len(set(targets_sym))} unique targets | {shared_targets} shared with pySCENIC | {len(set(tfs_sym))} TFs | {(~not_self).sum()+not_self.sum():,} edges")

thresh=0.001: 289 unique targets | 206 shared with pySCENIC | 982 TFs | 39,926 edges
thresh=0.0005: 444 unique targets | 305 shared with pySCENIC | 982 TFs | 58,503 edges
thresh=0.0001: 952 unique targets | 672 shared with pySCENIC | 982 TFs | 129,618 edges
thresh=5e-05: 1154 unique targets | 834 shared with pySCENIC | 982 TFs | 176,082 edges


In [ ]:
for thresh in [0.001, 0.0005, 0.0001, 0.00005]:
    row_idxs, col_idxs = np.where(mat > thresh)
    tf_filter   = is_tf[row_idxs]
    tfs_sym     = symbols[row_idxs[tf_filter]]
    targets_sym = symbols[col_idxs[tf_filter]]
    not_self    = tfs_sym != targets_sym
    tfs_sym     = tfs_sym[not_self]
    targets_sym = targets_sym[not_self]

    edge_set = set(zip(tfs_sym, targets_sym))
    overlap  = len(edge_set & pyscenic_edge_set)
    pct      = overlap / len(pyscenic_edge_set) * 100

    print(f"thresh={thresh}: {len(edge_set):,} edges | overlap={overlap} ({pct:.2f}% of pySCENIC)")

thresh=0.001: 39,906 edges | overlap=96 (0.36% of pySCENIC)
thresh=0.0005: 58,473 edges | overlap=140 (0.52% of pySCENIC)
thresh=0.0001: 129,556 edges | overlap=275 (1.02% of pySCENIC)
thresh=5e-05: 175,999 edges | overlap=386 (1.43% of pySCENIC)


In [ ]:
overlap_stats = []

for state in STATES:
    if state not in scprint_edges_by_state:
        print(f'Skipping {state} (scPRINT2 file not found)')
        continue

    print(f'\n{"="*70}')
    print(f'Comparing {state}: pySCENIC vs. scPRINT2')
    print(f'{"="*70}')

    scprint_edges = scprint_edges_by_state[state]

    # Compute overlap
    shared_edges = pyscenic_edge_set & scprint_edges

    # TF overlap (which TFs are in both networks?)
    pyscenic_tfs = set([e[0] for e in pyscenic_edge_set])
    scprint_tfs = set([e[0] for e in scprint_edges])
    shared_tfs = pyscenic_tfs & scprint_tfs

    # Compute statistics
    n_scenic_edges = len(pyscenic_edge_set)
    n_scprint_edges = len(scprint_edges)
    n_shared = len(shared_edges)

    if n_scenic_edges > 0:
        pct_scenic_in_scprint = (n_shared / n_scenic_edges) * 100
    else:
        pct_scenic_in_scprint = 0

    if n_scprint_edges > 0:
        pct_scprint_in_scenic = (n_shared / n_scprint_edges) * 100
    else:
        pct_scprint_in_scenic = 0

    print(f'pySCENIC edges: {n_scenic_edges:,}')
    print(f'scPRINT2 edges: {n_scprint_edges:,}')
    print(f'Shared edges: {n_shared:,}')
    print(f'  {pct_scenic_in_scprint:.2f}% of pySCENIC edges in scPRINT2')
    print(f'  {pct_scprint_in_scenic:.2f}% of scPRINT2 edges in pySCENIC')
    print(f'\nTF coverage:')
    print(f'  pySCENIC TFs: {len(pyscenic_tfs)}')
    print(f'  scPRINT2 TFs: {len(scprint_tfs)}')
    print(f'  Shared TFs: {len(shared_tfs)} ({len(shared_tfs)/len(pyscenic_tfs)*100:.1f}% of pySCENIC TFs)')

    # Top 10 shared edges
    print(f'\nTop 10 shared edges:')
    shared_edges_list = sorted(list(shared_edges))[:10]
    for tf, target in shared_edges_list:
        print(f'  {tf} → {target}')

    # Collect stats
    overlap_stats.append({
        'cell_state': state,
        'n_pyscenic_edges': n_scenic_edges,
        'n_scprint2_edges': n_scprint_edges,
        'n_shared_edges': n_shared,
        'pct_pyscenic_in_scprint2': pct_scenic_in_scprint,
        'pct_scprint2_in_pyscenic': pct_scprint_in_scenic,
        'n_pyscenic_tfs': len(pyscenic_tfs),
        'n_scprint2_tfs': len(scprint_tfs),
        'n_shared_tfs': len(shared_tfs),
    })

print(f'\n{"="*70}')
print('Overlap computation complete')
print(f'{"="*70}')


Comparing CD4_rest: pySCENIC vs. scPRINT2
pySCENIC edges: 27,019
scPRINT2 edges: 9,470
Shared edges: 19
  0.07% of pySCENIC edges in scPRINT2
  0.20% of scPRINT2 edges in pySCENIC

TF coverage:
  pySCENIC TFs: 235
  scPRINT2 TFs: 982
  Shared TFs: 180 (76.6% of pySCENIC TFs)

Top 10 shared edges:
  ATF4 → CEBPG
  CEBPB → ASH1L
  CEBPB → CEBPG
  CREM → CEBPG
  CREM → RPS27
  CREM → VEZF1
  EOMES → CCL4
  FOSL1 → ASH1L
  FOSL2 → CEBPG
  IRF7 → B2M

Comparing CD4_act: pySCENIC vs. scPRINT2
pySCENIC edges: 27,019
scPRINT2 edges: 6,904
Shared edges: 16
  0.06% of pySCENIC edges in scPRINT2
  0.23% of scPRINT2 edges in pySCENIC

TF coverage:
  pySCENIC TFs: 235
  scPRINT2 TFs: 973
  Shared TFs: 181 (77.0% of pySCENIC TFs)

Top 10 shared edges:
  CEBPB → ASH1L
  FOSL1 → ASH1L
  FOSL2 → ZNF232
  HOXB2 → ZNF232
  IRF7 → B2M
  IRF7 → RPS27
  JUNB → RPS27
  KLF3 → ASH1L
  KLF6 → ASH1L
  KLF6 → ZNF232

Comparing CD8_rest: pySCENIC vs. scPRINT2
pySCENIC edges: 27,019
scPRINT2 edges: 7,873
Shared e

## Step 4: Save Results

In [ ]:
# Save overlap statistics
overlap_df = pd.DataFrame(overlap_stats)

print('\n=== Edge Overlap Summary (pySCENIC vs. scPRINT2) ===')
print(overlap_df[['cell_state', 'n_pyscenic_edges', 'n_scprint2_edges',
                   'n_shared_edges', 'pct_pyscenic_in_scprint2', 'pct_scprint2_in_pyscenic']].to_string(index=False))

output_path = os.path.join(DRIVE_ANALYSIS, 'edge_overlap_scenic_vs_scprint2.csv')
overlap_df.to_csv(output_path, index=False)
print(f'\n✓ Saved: {output_path}')

# Also save TF agreement
tf_agreement = overlap_df[['cell_state', 'n_pyscenic_tfs', 'n_scprint2_tfs', 'n_shared_tfs']]
tf_agreement['pct_pyscenic_tfs_shared'] = (tf_agreement['n_shared_tfs'] / tf_agreement['n_pyscenic_tfs'] * 100).round(2)

print('\n=== TF Agreement Summary ===')
print(tf_agreement.to_string(index=False))

tf_path = os.path.join(DRIVE_ANALYSIS, 'tf_agreement_scenic_vs_scprint2.csv')
tf_agreement.to_csv(tf_path, index=False)
print(f'\n✓ Saved: {tf_path}')


=== Edge Overlap Summary (pySCENIC vs. scPRINT2) ===
cell_state  n_pyscenic_edges  n_scprint2_edges  n_shared_edges  pct_pyscenic_in_scprint2  pct_scprint2_in_pyscenic
  CD4_rest             27019              9470              19                  0.070321                  0.200634
   CD4_act             27019              6904              16                  0.059218                  0.231750
  CD8_rest             27019              7873              23                  0.085125                  0.292138
   CD8_act             27019              8446              22                  0.081424                  0.260478

✓ Saved: /content/drive/MyDrive/benchmarking_paper/datasets/analysis_outputs/may20/edge_overlap_scenic_vs_scprint2.csv

=== TF Agreement Summary ===
cell_state  n_pyscenic_tfs  n_scprint2_tfs  n_shared_tfs  pct_pyscenic_tfs_shared
  CD4_rest             235             982           180                    76.60
   CD4_act             235             973           181 

/tmp/ipykernel_6150/539381215.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tf_agreement['pct_pyscenic_tfs_shared'] = (tf_agreement['n_shared_tfs'] / tf_agreement['n_pyscenic_tfs'] * 100).round(2)


In [ ]:
print("=" * 70)
print("WHAT DO pySCENIC AND scPRINT2 AGREE ON?")
print("=" * 70)

for state in STATES:
    if state not in scprint_edges_by_state:
        continue

    scprint_edges = scprint_edges_by_state[state]
    shared_edges  = pyscenic_edge_set & scprint_edges
    shared_tfs    = set(e[0] for e in pyscenic_edge_set) & set(e[0] for e in scprint_edges)
    only_scenic   = set(e[0] for e in pyscenic_edge_set) - set(e[0] for e in scprint_edges)
    only_scprint  = set(e[0] for e in scprint_edges) - set(e[0] for e in pyscenic_edge_set)

    print(f"\n{'─'*70}")
    print(f"  {state}")
    print(f"{'─'*70}")

    print(f"\n  EDGES (TF → target pairs):")
    print(f"    pySCENIC has:          {len(pyscenic_edge_set):>7,} edges")
    print(f"    scPRINT2 has:          {len(scprint_edges):>7,} edges")
    print(f"    Both agree on:         {len(shared_edges):>7,} edges  ← same TF→target pair in both")
    print(f"    pySCENIC only:         {len(pyscenic_edge_set)-len(shared_edges):>7,} edges  ← motif evidence only")
    print(f"    scPRINT2 only:         {len(scprint_edges)-len(shared_edges):>7,} edges  ← attention evidence only")

    print(f"\n  TFs (regulators):")
    print(f"    pySCENIC has:          {len(set(e[0] for e in pyscenic_edge_set)):>7} TFs")
    print(f"    scPRINT2 has:          {len(set(e[0] for e in scprint_edges)):>7} TFs")
    print(f"    Both agree on:         {len(shared_tfs):>7} TFs  ← same TF active in both")
    print(f"    pySCENIC only:         {len(only_scenic):>7} TFs")
    print(f"    scPRINT2 only:         {len(only_scprint):>7} TFs")

    print(f"\n  SHARED EDGES (highest confidence — supported by motif + attention):")
    for tf, target in sorted(shared_edges):
        print(f"    {tf:15} →  {target}")

    print(f"\n  SHARED TFs (active in both methods):")
    print(f"    {', '.join(sorted(shared_tfs)[:30])}")
    if len(shared_tfs) > 30:
        print(f"    ... and {len(shared_tfs)-30} more")

print(f"\n{'='*70}")
print("SUMMARY ACROSS ALL STATES")
print(f"{'='*70}")
print(f"  Edge overlap is LOW (~0.07%) — expected, methods use different evidence")
print(f"  TF overlap is HIGH (~76%) — both methods identify the same regulators")
print(f"  Shared edges = highest confidence TF→target pairs for manuscript")


WHAT DO pySCENIC AND scPRINT2 AGREE ON?

──────────────────────────────────────────────────────────────────────
  CD4_rest
──────────────────────────────────────────────────────────────────────

  EDGES (TF → target pairs):
    pySCENIC has:           27,019 edges
    scPRINT2 has:            9,470 edges
    Both agree on:              19 edges  ← same TF→target pair in both
    pySCENIC only:          27,000 edges  ← motif evidence only
    scPRINT2 only:           9,451 edges  ← attention evidence only

  TFs (regulators):
    pySCENIC has:              235 TFs
    scPRINT2 has:              982 TFs
    Both agree on:             180 TFs  ← same TF active in both
    pySCENIC only:              55 TFs
    scPRINT2 only:             802 TFs

  SHARED EDGES (highest confidence — supported by motif + attention):
    ATF4            →  CEBPG
    CEBPB           →  ASH1L
    CEBPB           →  CEBPG
    CREM            →  CEBPG
    CREM            →  RPS27
    CREM            →  VEZF1
   

## Summary

**Task 2.1 complete.** Edge overlap analysis shows method concordance per cell state.

**Next steps:**
- If overlap is low (<20%): investigate differences in methodology (threshold, TF list, gene filtering)
- If overlap is good (>20%): proceed to Task 2.2 (conserved vs. rewired edges)

In [ ]:
# Check columns of your perturb-net files
import pandas as pd

PERTURB_DIR = '/content/drive/MyDrive/benchmarking_paper/datasets/perturbnet_outputs'

df_rest = pd.read_csv(f'{PERTURB_DIR}/perturb_net_tfs_only_Rest_filtered.csv')
print("Rest columns:", df_rest.columns.tolist())
print(df_rest.head(3))

Rest columns: ['source_TF', 'target_gene', 'log2FC', 'adj_p_value', 'p_value', 'zscore', 'culture_condition', 'n_guides_averaged']
  source_TF target_gene    log2FC   adj_p_value       p_value    zscore  \
0      ADNP       ALG13  0.457487  2.706116e-03  1.750398e-06  4.780293   
1      ADNP        BIN2 -0.325718  1.971065e-02  2.231154e-05 -4.240406   
2      ADNP      GNPDA1 -0.894893  4.478396e-07  1.086286e-10 -6.454428   

  culture_condition  n_guides_averaged  
0              Rest                  1  
1              Rest                  1  
2              Rest                  1  


In [ ]:
PERTURB_DIR = '/content/drive/MyDrive/benchmarking_paper/datasets/perturbnet_outputs'

# Load perturb-seq networks
perturb_rest    = pd.read_csv(f'{PERTURB_DIR}/perturb_net_tfs_only_Rest_filtered.csv')
perturb_stim8   = pd.read_csv(f'{PERTURB_DIR}/perturb_net_tfs_only_Stim8hr_filtered.csv')
perturb_stim48  = pd.read_csv(f'{PERTURB_DIR}/perturb_net_tfs_only_Stim48hr_filtered.csv')

# Build edge sets
perturb_edge_sets = {
    'Rest':    set(zip(perturb_rest['source_TF'],   perturb_rest['target_gene'])),
    'Stim8hr': set(zip(perturb_stim8['source_TF'],  perturb_stim8['target_gene'])),
    'Stim48hr':set(zip(perturb_stim48['source_TF'], perturb_stim48['target_gene'])),
}

# Build TF sets
perturb_tf_sets = {
    'Rest':    set(perturb_rest['source_TF']),
    'Stim8hr': set(perturb_stim8['source_TF']),
    'Stim48hr':set(perturb_stim48['source_TF']),
}

print("Perturb-seq network sizes:")
for k, v in perturb_edge_sets.items():
    print(f"  {k}: {len(v):,} edges | {len(perturb_tf_sets[k])} TFs")

# Map cell states to perturb-seq conditions
state_to_perturb = {
    'CD4_rest': ['Rest'],
    'CD4_act':  ['Stim8hr', 'Stim48hr'],
    'CD8_rest': ['Rest'],
    'CD8_act':  ['Stim8hr', 'Stim48hr'],
}

def jaccard(a, b):
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)

print("\n" + "="*70)
print("3-WAY COMPARISON: scPRINT2 vs pySCENIC vs Perturb-seq")
print("="*70)

summary_rows = []

for state in STATES:
    scprint_edges  = scprint_edges_by_state[state]   # TF-level edges
    scprint_tfs    = set(e[0] for e in scprint_edges)
    scenic_tfs     = set(e[0] for e in pyscenic_edge_set)
    perturb_conds  = state_to_perturb[state]

    print(f"\n{'─'*70}")
    print(f"  {state}")
    print(f"{'─'*70}")

    for pcond in perturb_conds:
        perturb_edges = perturb_edge_sets[pcond]
        perturb_tfs   = perturb_tf_sets[pcond]

        # Edge overlaps
        sp_sc  = scprint_edges & pyscenic_edge_set
        sp_ps  = scprint_edges & perturb_edges
        sc_ps  = pyscenic_edge_set & perturb_edges
        all3   = scprint_edges & pyscenic_edge_set & perturb_edges

        # TF overlaps
        sp_sc_tf = scprint_tfs & scenic_tfs
        sp_ps_tf = scprint_tfs & perturb_tfs
        sc_ps_tf = scenic_tfs  & perturb_tfs
        all3_tf  = scprint_tfs & scenic_tfs & perturb_tfs

        print(f"\n  vs Perturb-seq [{pcond}]:")
        print(f"\n  EDGES:")
        print(f"    scPRINT2:                {len(scprint_edges):>7,}")
        print(f"    pySCENIC:                {len(pyscenic_edge_set):>7,}")
        print(f"    Perturb-seq:             {len(perturb_edges):>7,}")
        print(f"    scPRINT2 ∩ pySCENIC:    {len(sp_sc):>7,}  (Jaccard: {jaccard(scprint_edges, pyscenic_edge_set):.4f})")
        print(f"    scPRINT2 ∩ Perturb-seq: {len(sp_ps):>7,}  (Jaccard: {jaccard(scprint_edges, perturb_edges):.4f})")
        print(f"    pySCENIC ∩ Perturb-seq: {len(sc_ps):>7,}  (Jaccard: {jaccard(pyscenic_edge_set, perturb_edges):.4f})")
        print(f"    ALL THREE:               {len(all3):>7,}")

        print(f"\n  TFs:")
        print(f"    scPRINT2:                {len(scprint_tfs):>5}")
        print(f"    pySCENIC:                {len(scenic_tfs):>5}")
        print(f"    Perturb-seq:             {len(perturb_tfs):>5}")
        print(f"    scPRINT2 ∩ pySCENIC:    {len(sp_sc_tf):>5}")
        print(f"    scPRINT2 ∩ Perturb-seq: {len(sp_ps_tf):>5}")
        print(f"    pySCENIC ∩ Perturb-seq: {len(sc_ps_tf):>5}")
        print(f"    ALL THREE:               {len(all3_tf):>5}")

        if all3:
            print(f"\n  Edges in ALL THREE networks:")
            for tf, tgt in sorted(all3):
                print(f"    {tf:15} → {tgt}")

        if all3_tf:
            print(f"\n  TFs in ALL THREE networks:")
            print(f"    {', '.join(sorted(all3_tf))}")

        summary_rows.append({
            'state': state, 'perturb_condition': pcond,
            'n_scprint_edges': len(scprint_edges),
            'n_scenic_edges': len(pyscenic_edge_set),
            'n_perturb_edges': len(perturb_edges),
            'scprint_scenic_overlap': len(sp_sc),
            'scprint_perturb_overlap': len(sp_ps),
            'scenic_perturb_overlap': len(sc_ps),
            'all_three_overlap': len(all3),
            'jaccard_scprint_scenic': jaccard(scprint_edges, pyscenic_edge_set),
            'jaccard_scprint_perturb': jaccard(scprint_edges, perturb_edges),
            'jaccard_scenic_perturb': jaccard(pyscenic_edge_set, perturb_edges),
            'n_scprint_tfs': len(scprint_tfs),
            'n_scenic_tfs': len(scenic_tfs),
            'n_perturb_tfs': len(perturb_tfs),
            'all_three_tfs': len(all3_tf),
        })

summary_df = pd.DataFrame(summary_rows)
out = os.path.join(DRIVE_ANALYSIS, 'threeway_comparison.csv')
summary_df.to_csv(out, index=False)
print(f"\n{'='*70}")
print(f"Saved summary → {out}")

Perturb-seq network sizes:
  Rest: 28,836 edges | 409 TFs
  Stim8hr: 33,569 edges | 429 TFs
  Stim48hr: 28,521 edges | 424 TFs

3-WAY COMPARISON: scPRINT2 vs pySCENIC vs Perturb-seq

──────────────────────────────────────────────────────────────────────
  CD4_rest
──────────────────────────────────────────────────────────────────────

  vs Perturb-seq [Rest]:

  EDGES:
    scPRINT2:                  9,470
    pySCENIC:                 27,019
    Perturb-seq:              28,836
    scPRINT2 ∩ pySCENIC:         19  (Jaccard: 0.0005)
    scPRINT2 ∩ Perturb-seq:      14  (Jaccard: 0.0004)
    pySCENIC ∩ Perturb-seq:     217  (Jaccard: 0.0039)
    ALL THREE:                     0

  TFs:
    scPRINT2:                  982
    pySCENIC:                  235
    Perturb-seq:               409
    scPRINT2 ∩ pySCENIC:      180
    scPRINT2 ∩ Perturb-seq:   387
    pySCENIC ∩ Perturb-seq:    85
    ALL THREE:                  85

  TFs in ALL THREE networks:
    ARID3A, ATF1, ATF4, ATF6B, BACH

In [ ]:
import pandas as pd
import numpy as np

def make_overlap_matrix(tf_sets_dict):
    """Build TF overlap count and Jaccard matrices from a dict of {label: set_of_TFs}"""
    labels = list(tf_sets_dict.keys())
    n = len(labels)
    count_mat   = np.zeros((n, n), dtype=int)
    jaccard_mat = np.zeros((n, n))

    for i, l1 in enumerate(labels):
        for j, l2 in enumerate(labels):
            a, b = tf_sets_dict[l1], tf_sets_dict[l2]
            if i == j:
                count_mat[i, j] = len(a)  # diagonal = total TFs
            else:
                count_mat[i, j] = len(a & b)
            jaccard_mat[i, j] = len(a & b) / len(a | b) if (a | b) else 0.0

    count_df   = pd.DataFrame(count_mat,   index=labels, columns=labels)
    jaccard_df = pd.DataFrame(jaccard_mat, index=labels, columns=labels)
    return count_df, jaccard_df


def print_overlap_table(state, scprint_tfs, scenic_tfs, perturb_tf_sets, perturb_conds):
    print(f"\n--- {state} ---")

    tf_sets = {f'scPRINT2_{state}': scprint_tfs,
               f'pySCENIC_{state}': scenic_tfs}
    for cond in perturb_conds:
        tf_sets[f'Perturb-seq_{state}_{cond}'] = perturb_tf_sets[cond]

    count_df, jaccard_df = make_overlap_matrix(tf_sets)

    print("\n  Overlap counts (diagonal = total TFs in that method-state):")
    print(count_df.to_string())
    print("\n  Jaccard similarity:")
    print(jaccard_df.round(4).to_string())

    return count_df, jaccard_df


# State → Perturb-seq condition mapping
state_to_perturb = {
    'CD4_rest': ['Rest'],
    'CD4_act':  ['Stim8hr', 'Stim48hr'],
    'CD8_rest': ['Rest'],
    'CD8_act':  ['Stim8hr', 'Stim48hr'],
}

scenic_tfs = set(e[0] for e in pyscenic_edge_set)
all_count_dfs   = {}
all_jaccard_dfs = {}

for state in STATES:
    scprint_tfs  = set(e[0] for e in scprint_edges_by_state[state])
    perturb_conds = state_to_perturb[state]

    count_df, jaccard_df = print_overlap_table(
        state, scprint_tfs, scenic_tfs, perturb_tf_sets, perturb_conds
    )
    all_count_dfs[state]   = count_df
    all_jaccard_dfs[state] = jaccard_df

# Save to Drive
for state in STATES:
    all_count_dfs[state].to_csv(
        os.path.join(DRIVE_ANALYSIS, f'{state}_tf_overlap_counts.csv'))
    all_jaccard_dfs[state].to_csv(
        os.path.join(DRIVE_ANALYSIS, f'{state}_tf_jaccard.csv'))

print("\nSaved all matrices to Drive ✓")


--- CD4_rest ---

  Overlap counts (diagonal = total TFs in that method-state):
                           scPRINT2_CD4_rest  pySCENIC_CD4_rest  Perturb-seq_CD4_rest_Rest
scPRINT2_CD4_rest                        982                180                        387
pySCENIC_CD4_rest                        180                235                         85
Perturb-seq_CD4_rest_Rest                387                 85                        409

  Jaccard similarity:
                           scPRINT2_CD4_rest  pySCENIC_CD4_rest  Perturb-seq_CD4_rest_Rest
scPRINT2_CD4_rest                     1.0000             0.1736                     0.3855
pySCENIC_CD4_rest                     0.1736             1.0000                     0.1521
Perturb-seq_CD4_rest_Rest             0.3855             0.1521                     1.0000

--- CD4_act ---

  Overlap counts (diagonal = total TFs in that method-state):
                              scPRINT2_CD4_act  pySCENIC_CD4_act  Perturb-seq_CD4_act_St